# Tale Role — mechanics runner (Colab)

**One-time:** Colab Secrets → `HF_TOKEN`. Optional: `RENDER_API_KEY` + `RENDER_SERVICE_ID`.

Open in a **second** tab while storyteller runs in the first.

1. Runtime → **T4 GPU**
2. **Runtime → Run all**

In [ ]:
!pip install -q "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" huggingface_hub requests
!wget -q https://raw.githubusercontent.com/leventkok/tale-role/main/services/llm-runner/serve.py
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
from pathlib import Path
assert Path("serve.py").exists(), "serve.py download failed — check network"
print("serve.py ready")

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_LOAD_IN_4BIT"] = "1"
os.environ["HF_MODEL_ID"] = "levonov/talerole-mechanics"
print("ready", os.environ["HF_MODEL_ID"])

In [ ]:
import os, subprocess, sys, time, re

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
server = subprocess.Popen(
    [sys.executable, "serve.py", "--role", "mechanics", "--hf-model", os.environ["HF_MODEL_ID"], "--port", "8092"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
print("loading model from Hub (first run can take 10+ min)...")
deadline = time.time() + 900
while time.time() < deadline:
    line = server.stdout.readline()
    if line:
        print(line, end="")
        if "llm-runner" in line:
            break
    elif server.poll() is not None:
        print(server.stdout.read())
        raise SystemExit("runner exited")
    else:
        time.sleep(1)
else:
    raise SystemExit("runner timeout — model still loading; retry or check HF_TOKEN")

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8092"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if not line:
        continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print("\nPUBLIC URL:", url)
print("health:", (url or "") + "/health/live")

In [ ]:
import requests
from google.colab import userdata

ENV_KEY = "LLM_MECHANICS_URL"

if not url:
    raise SystemExit("no PUBLIC URL — fix tunnel cell first")

health = requests.get(url + "/health/live", timeout=30).json()
print("health/live:", health)
if not health.get("weights"):
    print("warning: weights not ready yet — wait and re-run this cell")

try:
    api_key = userdata.get("RENDER_API_KEY")
    service_id = userdata.get("RENDER_SERVICE_ID")
except Exception:
    print(f"manual Render env: {ENV_KEY}={url}")
else:
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    r = requests.put(
        f"https://api.render.com/v1/services/{service_id}/env-vars/{ENV_KEY}",
        headers=headers,
        json={"value": url},
        timeout=60,
    )
    r.raise_for_status()
    d = requests.post(
        f"https://api.render.com/v1/services/{service_id}/deploys",
        headers=headers,
        json={},
        timeout=60,
    )
    d.raise_for_status()
    print(f"Render {ENV_KEY} updated + deploy triggered")